<h1>PPDB Export Examples</h1>

<h2>Imports</h2>

In [3]:
from pathlib import Path
import io
import requests
import time

from astropy.table import Table
from pyvo.dal import AsyncTAPJob, TAPService
from pyvo.dal.tap import TAPService
import pandas as pd
import pyvo

from lsst.rsp import RSPClient, get_tap_service, get_service_url, get_access_token

<h2>Service setup</h2>

Get the PPDB TAP service.

In [4]:
url = get_service_url("tap", "prompt")

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {get_access_token()}"
})
service = TAPService(url, session=session)

<h2>Export utility function</h2>

This method will query a specific PPDB table using the TAP service to determine which days have data and what are the expected record counts. The data will then be exported to a set of parquet files, one per day. If a parquet file already exists with the expected number of records in the output directory, the export for that day will be skipped unless `skip_existing` is set to `False`.

In [ ]:
def export_ppdb_table(
    table_name: str,
    export_dir: str = "ppdb_export_data",
    skip_existing: bool = True
):
    """Export PPDB table data to parquet files, one per day.

    Days will be skipped if there is a parquet file already present 
    with the correct record count.

    Parameters
    ----------
    table_name
        Name of table to export such as "DiaObject" or "DiaSource".
    export_dir
        Directory where parquet files should be written.
        Defaults to ``ppdb_export_data.
    """
    print(f"Starting export of {table_name} table...\n")
    export_start = time.time()
    
    # Create export directory (or use existing).
    Path(export_dir).mkdir(exist_ok=True)

    # Determine which timing column to use for the table.
    if table_name == "DiaObject":
        ts_col = "validityStartMjdTai"
    elif table_name == "DiaSource" or table_name == "DiaForcedSource":
        ts_col = "midpointMjdTai"
    else:
        raise Exception(f"Unsupported table: {table_name}")

    # Get a list of days (MJD TAI format) which have data.
    sql = f"""
        SELECT FLOOR({ts_col}) AS day_mjd_tai,
            COUNT(*) AS record_count
        FROM ppdb.{table_name}
        GROUP BY day_mjd_tai ORDER BY day_mjd_tai
        """
    job = service.submit_job(sql)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    if job.phase == "ERROR":
        job.raise_if_error() 
    days_result = job.fetch_result().to_table()
    
    days = [d for d in days_result["day_mjd_tai"]]
    record_counts = [c for c in days_result["record_count"]]
    
    # Loop over the days with data and process them.
    for day, record_count in zip(days, record_counts, strict=True):

        print(f"Processing day: {day}")
        print(f"  Expected record count: {record_count}")

        day_start = time.time()
        
        # Make directory for this day.
        output_dir = Path(export_dir, str(int(day)))
        output_dir.mkdir(exist_ok=True)
        output_path = output_dir / f"{table_name}.parquet"
    
        # Check for and verify an existing output file and skip if exists
        # with correct record count.
        if skip_existing and output_path.exists():
            print(f"  Parquet file already exists: {output_path}")
            try:
                df = pd.read_parquet(output_path)
                parquet_record_count = len(df)
                print(f"  Existing parquet file has {parquet_record_count} records.")
                if parquet_record_count == record_count:
                    print("  Skipping this day - parquet file with correct record count already exists.\n")
                    continue
                else:
                    print(f"  Record count mismatch: {parquet_record_count} != {record_count}")
                    print("  File will be recreated.")
            except Exception as e:
                # This probably indicates an invalid or partially written parquet file.
                print(e)
        
        # Get data for the day from the TAP service.
        sql = f"SELECT * FROM ppdb.{table_name} WHERE FLOOR({ts_col}) = {day}"
        print(f"  Executing SQL: {sql}")
         
        # Run the SQL job.
        job_start = time.time()
        job = AsyncTAPJob.create(
            service.baseurl,
            sql,
            RESPONSEFORMAT="application/vnd.apache.parquet",
            session=session
        )
        job = job.run().wait()
        job_end = time.time()
        job_elapsed = job_end - job_start
        print(f"  Job took {job_elapsed:.2f} seconds")

        # Fetch the data.
        fetch_start = time.time()
        response = session.get(job.result_uri, stream=True)
        table_data = Table.read(io.BytesIO(response.content), format="parquet.votable")
        fetch_end = time.time() 
        fetch_elapsed = fetch_end - fetch_start
        print(f"  Fetch took {fetch_elapsed:.2f} seconds")
            
        # Write the entire day's data to a parquet file.
        parq_start = time.time()
        table_data.write(output_path, format="parquet", overwrite=True)
        parq_end = time.time()
        parq_elapsed = parq_end - parq_start
        print(f"  Wrote table data to '{output_path}' in {parq_elapsed:.2f} seconds")

        day_end = time.time()
        day_elapsed = day_end - day_start
        print(f"  Exported data from day {day} in {day_elapsed:.2f} seconds\n")

    export_end = time.time()
    export_elapsed = export_end - export_start
    print(f"Export of {table_name} completed in {export_elapsed:.0f} seconds.")

In [ ]:
def export_ppdb_by_day(
    tables: list[str] | None = None,
    export_dir: str | Path = "ppdb_export_data",
    skip_existing: bool = True,
) -> None:
    """Export PPDB table data to parquet files, grouped by day.

    Data are exported into one directory per MJD TAI day, with one parquet
    file per table inside each day directory. For example::

        ppdb_export_data/
            60400/
                DiaObject.parquet
                DiaSource.parquet
                DiaForcedSource.parquet

    Days are skipped for a table if the corresponding parquet file already
    exists and has the expected record count.

    Parameters
    ----------
    tables
        Names of tables to export. If `None`, exports ``DiaObject``,
        ``DiaSource``, and ``DiaForcedSource``.
    export_dir
        Directory where parquet files should be written.
    skip_existing
        If `True`, skip existing parquet files when their record count matches
        the expected count from the TAP service.

    Returns
    -------
    None
        This function writes parquet files and prints progress information.
    """
    if tables is None:
        tables = ["DiaObject", "DiaSource", "DiaForcedSource"]

    timing_columns: dict[str, str] = {
        "DiaObject": "validityStartMjdTai",
        "DiaSource": "midpointMjdTai",
        "DiaForcedSource": "midpointMjdTai",
    }

    for table_name in tables:
        if table_name not in timing_columns:
            raise ValueError(f"Unsupported table: {table_name}")

    export_start = time.time()
    export_path = Path(export_dir)
    export_path.mkdir(exist_ok=True)

    print(f"Starting export of PPDB tables: {', '.join(tables)}\n")

    day_counts_by_table: dict[str, dict[int, int]] = {}
    all_days: set[int] = set()

    for table_name in tables:
        ts_col = timing_columns[table_name]

        print(f"Finding days with data for {table_name}...")

        sql = f"""
            SELECT
                FLOOR({ts_col}) AS day_mjd_tai,
                COUNT(*) AS record_count
            FROM ppdb.{table_name}
            GROUP BY day_mjd_tai
            ORDER BY day_mjd_tai
        """

        job = service.submit_job(sql)
        job.run()
        job.wait(phases=["COMPLETED", "ERROR"])

        if job.phase == "ERROR":
            job.raise_if_error()

        days_result = job.fetch_result().to_table()

        table_day_counts: dict[int, int] = {}

        for day, record_count in zip(
            days_result["day_mjd_tai"],
            days_result["record_count"],
            strict=True,
        ):
            day_int = int(day)
            count_int = int(record_count)

            table_day_counts[day_int] = count_int
            all_days.add(day_int)

        day_counts_by_table[table_name] = table_day_counts

        print(f"  Found {len(table_day_counts)} days with data.\n")

    if not all_days:
        print("No data found for requested tables.")
        return

    for day in sorted(all_days):
        print(f"Processing day: {day}")

        day_start = time.time()
        day_output_dir = export_path / str(day)
        day_output_dir.mkdir(exist_ok=True)

        for table_name in tables:
            table_day_counts = day_counts_by_table[table_name]

            if day not in table_day_counts:
                print(f"  {table_name}: no records for this day.")
                continue

            record_count = table_day_counts[day]
            output_path = day_output_dir / f"{table_name}.parquet"
            ts_col = timing_columns[table_name]

            print(f"  {table_name}:")
            print(f"    Expected record count: {record_count}")

            if skip_existing and output_path.exists():
                print(f"    Parquet file already exists: {output_path}")

                try:
                    df = pd.read_parquet(output_path)
                    parquet_record_count = len(df)
                    print(f"    Existing parquet file has {parquet_record_count} records.")

                    if parquet_record_count == record_count:
                        print("    Skipping - parquet file with correct record count already exists.")
                        continue

                    print(f"    Record count mismatch: {parquet_record_count} != {record_count}")
                    print("    File will be recreated.")

                except Exception as e:
                    print(f"    Existing parquet file could not be read: {e}")
                    print("    File will be recreated.")

            sql = f"""
                SELECT *
                FROM ppdb.{table_name}
                WHERE FLOOR({ts_col}) = {day}
            """

            print(f"    Executing SQL: {sql.strip()}")

            job_start = time.time()

            job = AsyncTAPJob.create(
                service.baseurl,
                sql,
                RESPONSEFORMAT="application/vnd.apache.parquet",
                session=session,
            )

            job = job.run().wait()

            job_elapsed = time.time() - job_start
            print(f"    Job took {job_elapsed:.2f} seconds")

            fetch_start = time.time()

            response = session.get(job.result_uri, stream=True)
            response.raise_for_status()

            table_data = Table.read(
                io.BytesIO(response.content),
                format="parquet.votable",
            )

            fetch_elapsed = time.time() - fetch_start
            print(f"    Fetch took {fetch_elapsed:.2f} seconds")

            parq_start = time.time()

            table_data.write(
                output_path,
                format="parquet",
                overwrite=True,
            )

            parq_elapsed = time.time() - parq_start
            print(f"    Wrote table data to '{output_path}' in {parq_elapsed:.2f} seconds")

        day_elapsed = time.time() - day_start
        print(f"  Finished day {day} in {day_elapsed:.2f} seconds\n")

    export_elapsed = time.time() - export_start
    print(f"PPDB export completed in {export_elapsed:.0f} seconds.")

In [1]:
def _counts_by_day(
    tables: list[str] | None = None,
) -> dict[str, dict[int | None, int]]:
    """Print PPDB record counts per day by table.

    Parameters
    ----------
    tables
        Names of tables to inspect. If `None`, checks ``DiaObject``,
        ``DiaSource``, and ``DiaForcedSource``.

    Returns
    -------
    dict
        Mapping from table name to a mapping of MJD TAI day to record count.
        A day value of `None` means the timing column was `NULL`.
    """
    if tables is None:
        tables = ["DiaObject", "DiaSource", "DiaForcedSource"]

    timing_columns: dict[str, str] = {
        "DiaObject": "validityStartMjdTai",
        "DiaSource": "midpointMjdTai",
        "DiaForcedSource": "midpointMjdTai",
    }

    counts_by_table: dict[str, dict[int | None, int]] = {}
    all_days: set[int | None] = set()

    for table_name in tables:
        if table_name not in timing_columns:
            raise ValueError(f"Unsupported table: {table_name}")

        ts_col = timing_columns[table_name]

        sql = f"""
            SELECT
                FLOOR({ts_col}) AS day_mjd_tai,
                COUNT(*) AS record_count
            FROM ppdb.{table_name}
            GROUP BY day_mjd_tai
            ORDER BY day_mjd_tai
        """

        job = service.submit_job(sql)
        job.run()
        job.wait(phases=["COMPLETED", "ERROR"])

        if job.phase == "ERROR":
            job.raise_if_error()

        result = job.fetch_result().to_table()

        table_counts: dict[int | None, int] = {}

        for day, record_count in zip(
            result["day_mjd_tai"],
            result["record_count"],
            strict=True,
        ):
            if day is None or getattr(day, "mask", False):
                normalized_day = None
            else:
                normalized_day = int(day)

            table_counts[normalized_day] = int(record_count)
            all_days.add(normalized_day)

        counts_by_table[table_name] = table_counts

    sorted_days = sorted(day for day in all_days if day is not None)

    if None in all_days:
        sorted_days.append(None)

    table_width = max(len(table_name) for table_name in tables)
    day_width = max(len("day_mjd_tai"), *(len(str(day)) for day in sorted_days), len("NULL"))

    header = f"{'day_mjd_tai':>{day_width}}  " + "  ".join(
        f"{table_name:>{table_width}}" for table_name in tables
    )

    print(header)
    print("-" * len(header))

    for day in sorted_days:
        day_label = "NULL" if day is None else str(day)

        row = f"{day_label:>{day_width}}  " + "  ".join(
            f"{counts_by_table[table_name].get(day, 0):>{table_width}}"
            for table_name in tables
        )

        print(row)

    return counts_by_table

In [5]:
_counts_by_day()

day_mjd_tai        DiaObject        DiaSource  DiaForcedSource
--------------------------------------------------------------
      60973                0              414                0
      60975                0             1479                0
      60981                0             1348                0
      60983                0             2249                0
      60984                0             5966                0
      60985                0             4687                0
      60987                0             3027                0
      60988                0             6457                0
      60989                0             5265                0
      60990                0             1380                0
      60991                0             8477                0
      60994                0             5990                0
      60996                0             6812                0
      60998                0             4930          

{'DiaObject': {61083: 105094,
  61084: 1068078,
  61088: 36966,
  61090: 1853551,
  61091: 2440959,
  61092: 166495,
  61094: 1675433,
  61095: 2821560,
  61096: 666104,
  61097: 435843,
  61098: 252079,
  61099: 5173,
  61100: 2395,
  61101: 2450,
  61102: 108489,
  61103: 924,
  61104: 2893,
  61105: 3797,
  61106: 2591,
  61107: 2952,
  61108: 4769,
  61109: 1293,
  61129: 446,
  61132: 3238,
  61135: 2970,
  61136: 3626,
  61137: 7926,
  61138: 2190,
  61139: 6290,
  61140: 3220,
  61141: 1520,
  61142: 2687,
  61143: 4996,
  61144: 1003,
  61145: 5112,
  61146: 3277,
  61149: 519,
  61150: 875,
  61151: 8655,
  61152: 1752,
  61160: 4624,
  61161: 2415,
  61162: 2039,
  61172: 11303,
  61176: 2574,
  61177: 1168,
  61178: 12403,
  61179: 7763,
  61180: 1042,
  61182: 18452,
  61183: 95701,
  61184: 83903,
  61185: 50580,
  61186: 113262},
 'DiaSource': {60973: 414,
  60975: 1479,
  60981: 1348,
  60983: 2249,
  60984: 5966,
  60985: 4687,
  60987: 3027,
  60988: 6457,
  60989: 526

<h2>Export table data</h2>

In [ ]:
for table_name in ["DiaObject", "DiaSource", "DiaForcedSource"]:
    export_ppdb_table(table_name, skip_existing=True)

In [ ]:
export_ppdb_by_day(export_dir="ppdb_export_data_by_day", skip_existing=True)